# P2 — Crypto Inventory & Migration Planner

Phase 1 reproducible implementation.

## 1. Select the problem statement
P2 combines a classical cryptographic inventory and a measured A/B quantum processor benchmark.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.problem import score_inventory
from src.processors import load_processor
from src.routing import build_grover_circuit
from src.metrics import run_processor_benchmark, compare_metrics
from src.visualization import make_figures

## 2. Understand the inputs
The raw inventory has the required five columns. Processor JSON files contain architecture, basis-gate, port, and placeholder noise definitions.

In [ ]:
inventory = pd.read_csv(ROOT/'data/raw/inventory.csv')
processor_a = load_processor(ROOT/'processors/processor_A.json')
processor_b = load_processor(ROOT/'processors/processor_B.json')
display(inventory)
print(processor_a)
print(processor_b)

## 3. Build the inventory scorer

In [ ]:
scored = score_inventory(ROOT/'data/raw/inventory.csv', ROOT/'results/tables')
display(scored)

## 4. Build the logical Grover circuit

In [ ]:
logical = build_grover_circuit(marked='101', iterations=2, measure=True)
display(logical.draw('mpl'))

## 5. Processor A — transpile, execute, and measure

In [ ]:
row_a, compiled_a, baseline_a = run_processor_benchmark(logical, processor_a)
row_a

## 6. Processor B — same circuit and same settings

In [ ]:
row_b, compiled_b, baseline_b = run_processor_benchmark(logical, processor_b)
row_b

## 7. A vs B table

In [ ]:
comparison = pd.DataFrame(compare_metrics(row_a, row_b))
comparison.to_csv(ROOT/'results/tables/AB_comparison.csv', index=False)
comparison.to_csv(ROOT/'results/tables/final_comparison.csv', index=False)
pd.DataFrame([row_a]).to_csv(ROOT/'results/tables/processor_A_metrics.csv', index=False)
pd.DataFrame([row_b]).to_csv(ROOT/'results/tables/processor_B_metrics.csv', index=False)
comparison

## 8. Explain architecture effects
The coupling map determines how easily logical two-qubit interactions can be placed. A more connected map is not automatically better: the measured success probability also depends on the compiled gate sequence and the shared noise model. The A/B result table is the evidence used below.

In [ ]:
make_figures({'processor_A':processor_a, 'processor_B':processor_b}, logical, {'processor_A':compiled_a, 'processor_B':compiled_b}, [row_a,row_b], ROOT/'results/figures')
print('Figures written.')

## 9. Figures

In [ ]:
from IPython.display import Image, display
for name in ['processor_A_graph.png','processor_B_graph.png','logical_vs_transpiled.png','AB_performance.png','architecture_tradeoffs.png']:
    print(name); display(Image(filename=str(ROOT/'results/figures'/name)))

## 10. Reproducibility notes

In [ ]:
# Generate report and data-driven README results section.
report = "# P2 Report — Crypto Inventory & Migration Planner\n\n"
report += "## What changed?\nThe inventory was scored with the specified lookup tables, and each asset received a migration recommendation and phase. The same marked-state Grover circuit was compiled and executed on both placeholder processor definitions.\n\n"
report += f"## Which architectural property changed?\nProcessor A uses coupling edges {processor_a['coupling_map']}; Processor B uses coupling edges {processor_b['coupling_map']}. The basis gates and noise parameters were held constant.\n\n"
report += f"## Did transpilation add routing?\nUsing the all-to-all baseline, Processor A compiled to {row_a['two_qubit_gates']} counted two-qubit gates versus {row_a['baseline_two_qubit_gates']} baseline, for an estimated SWAP overhead of {row_a['swap_estimate']:.6f}. Processor B compiled to {row_b['two_qubit_gates']} versus {row_b['baseline_two_qubit_gates']} baseline, for an estimated SWAP overhead of {row_b['swap_estimate']:.6f}. This is an estimate because it divides extra two-qubit gates by three.\n\n"
report += f"## How many extra operations?\nThe total compiled operation counts were {row_a['total_ops']} for Processor A and {row_b['total_ops']} for Processor B; extra two-qubit counts versus baseline were {row_a['extra_two_qubit_gates']} and {row_b['extra_two_qubit_gates']}.\n\n"
report += f"## Did the problem metric change?\nThe measured success probabilities for state `101` were {row_a['success_probability']:.6f} ± {row_a['std_error']:.6f} for Processor A and {row_b['success_probability']:.6f} ± {row_b['std_error']:.6f} for Processor B. The difference (B − A) was {row_b['success_probability']-row_a['success_probability']:.6f}. Ideal reference probabilities were {row_a['ideal_success_probability']:.6f} and {row_b['ideal_success_probability']:.6f}.\n\n"
report += "## Could gate-set/noise differences also contribute?\nYes. This run keeps basis gates and the default placeholder noise model identical, but transpilation can produce different gate counts and layouts. If official processor data changes basis gates, ports, calibration, or noise, those factors can contribute and must be reported separately.\n\n"
report += "## What remains uncertain?\nThe processor definitions and noise values are placeholders until Challenge Kit values are supplied. Runtime is environment-dependent. The SWAP value is an estimate rather than a direct decomposition count.\n\n"
report += "## Limitations\nA toy three-qubit Grover run does not estimate the cost of attacking real RSA/ECC keys. It is an educational architecture comparison, not a cryptanalytic forecast.\n"
(ROOT/'report.md').write_text(report, encoding='utf-8')
readme = (ROOT/'README.md').read_text(encoding='utf-8')
key = f"## Key results\nThe executed benchmark measured success probabilities of {row_a['success_probability']:.6f} for Processor A and {row_b['success_probability']:.6f} for Processor B. Compiled depths were {row_a['depth']} and {row_b['depth']}; counted two-qubit gates were {row_a['two_qubit_gates']} and {row_b['two_qubit_gates']}. See `report.md` and the CSV tables for the complete measured evidence.\n"
readme = readme.replace('## Key results\nThis section is generated by the executed notebook from the result CSVs; it intentionally does not contain hand-entered measurements.\n', key)
(ROOT/'README.md').write_text(readme, encoding='utf-8')
print(report)
